In [33]:
import pandas as pd

In [34]:
df = pd.read_csv("./sample_data/IMDB Dataset.csv")

In [35]:
df.shape

(50000, 2)

In [36]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [37]:
df.isnull().sum()

,0
review,0
sentiment,0


In [38]:
df.drop_duplicates(inplace=True)

In [39]:
df.shape

(49582, 2)

## Pre-Processing

In [40]:
# 1. converting to lowercase

df['review'] = df['review'].str.lower()

In [41]:
# 2. removing urls
import re

def remove_urls(text):
    text = re.sub(r"https\S+","", text)

    return text

df['review'] = df['review'].apply(remove_urls)


In [42]:
df['review']

,review
0,one of the other reviewers has mentioned that ...
1,a wonderful little production. <br /><br />the...
2,i thought this was a wonderful way to spend ti...
3,basically there's a family where a little boy ...
4,"petter mattei's ""love in the time of money"" is..."
...,...
49995,i thought this movie did a down right good job...
49996,"bad plot, bad dialogue, bad acting, idiotic di..."
49997,i am a catholic taught in parochial elementary...
49998,i'm going to have to disagree with the previou...


In [43]:
#3) removing html tags

def remove_html_tags(text):
    text = re.sub("<.*?>", "", text)

    return text

df['review'] = df['review'].apply(remove_html_tags)

In [44]:

df['review']

,review
0,one of the other reviewers has mentioned that ...
1,a wonderful little production. the filming tec...
2,i thought this was a wonderful way to spend ti...
3,basically there's a family where a little boy ...
4,"petter mattei's ""love in the time of money"" is..."
...,...
49995,i thought this movie did a down right good job...
49996,"bad plot, bad dialogue, bad acting, idiotic di..."
49997,i am a catholic taught in parochial elementary...
49998,i'm going to have to disagree with the previou...


In [45]:
#4) remove punctations

def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]", "", text)

    return text

df['review'] = df['review'].apply(remove_punctuations)

In [46]:
df['review']

,review
0,one of the other reviewers has mentioned that ...
1,a wonderful little production the filming tech...
2,i thought this was a wonderful way to spend ti...
3,basically theres a family where a little boy j...
4,petter matteis love in the time of money is a ...
...,...
49995,i thought this movie did a down right good job...
49996,bad plot bad dialogue bad acting idiotic direc...
49997,i am a catholic taught in parochial elementary...
49998,im going to have to disagree with the previous...


In [47]:
# 5) removing the stopwards
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [48]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords


def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word, "")

    return text


df['review'] = df['review'].apply(remove_stopwords)



In [49]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti filming technique un...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


In [50]:
## 6. Stemming

# running - run
#playing - play

from nltk.stem import PorterStemmer

stemmer = PorterStemmer()


In [51]:
def stemming(text):

    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)

    for token in tokens:
        stemmed_token =ps.stem(token)
        stemmed_words.append(stemmed_token)

    text = " ".join(stemmed_words)

    return text

df['review'] = df['review'].apply(stemming)

In [52]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti film techniqu unssum oldtim...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


In [53]:
# Encoding

from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

df['sentiment'] = encoder.fit_transform(df['sentiment'])

In [54]:
y = df['sentiment']

In [55]:
y

,sentiment
0,1
1,1
2,1
3,0
4,1
...,...
49995,1
49996,0
49997,0
49998,0


In [56]:
# 8. vectorization


from sklearn.feature_extraction.text import TfidfVectorizer


tf = TfidfVectorizer(max_features=10000)

x = tf.fit_transform(df['review'])

In [57]:
x

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4365851 stored elements and shape (49582, 10000)>

## DataSet and Data Loader


In [58]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [59]:
x_train.shape

(39665, 10000)

In [60]:
x_test.shape

(9917, 10000)

In [61]:

from torch.utils.data import Dataset, DataLoader, TensorDataset
import torch

x_train = x_train.toarray()
x_test = x_test.toarray()

train_dataset = TensorDataset(torch.from_numpy(x_train).float(),
                              torch.from_numpy(y_train.values).float())

test_dataset = TensorDataset(torch.from_numpy(x_test).float(),
                             torch.from_numpy(y_test.values).float())



In [62]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

##Build our RNN

In [63]:
import torch.nn as nn
import torch.optim as optim

In [71]:
class RNN(nn.Module):
  def __init__(self, input_size, hidden_size=128, num_layers=1):
    super(RNN, self).__init__()

    self.hidden_size = hidden_size
    self.num_layers = num_layers

## RNN layer
    self.nn = nn.RNN(input_size, hidden_size=128, num_layers=1, batch_first=True)

    ## fully connected layer
    self.fc= nn.Linear(hidden_size, 1)

  def forward(self, x):
    h0= torch.zeros(self.num_layers, x.size(0), self.hidden_size)

    out , _ = self.nn(x, h0)


    out = self.fc(out[:,-1,:])

    return out

In [72]:
input_size = x_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

## Training the RNN

In [74]:
epochs =10

for epoch in range(epochs):
  model.train()

  for Xb, yb in train_loader:
    optimizer.zero_grad()

    Xb = Xb.unsqueeze(1)


    outputs = model(Xb)

    outputs = torch.sigmoid(outputs.squeeze())

    loss = criterion(outputs, yb)

    loss.backward()

    optimizer.step()

print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 10/10 and loss = 0.007692480925470591


In [76]:
# Evaluate

model.eval()
with torch.no_grad():
  correct = 0
  total = 0

  for Xb, yb in test_loader:
    Xb = Xb.unsqueeze(1)

    outputs = model(Xb)
    predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

    total += yb.size(0)
    correct += (predicted == yb).sum().item()

  accuracy = correct / total * 100
  print(f"Accuracy: {accuracy}")

Accuracy: 83.15014621357265
